In [1]:
import json
import torch
import torch.nn as nn
import math
from huggingface_hub import login, hf_hub_download
from datasets import Dataset
from transformers import PreTrainedTokenizerFast, TrainingArguments, Trainer, DataCollatorForTokenClassification
from safetensors.torch import load_file


In [2]:
!pip install --upgrade torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 71.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 38.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [ ]:
login(token=os.environ["HF_TOKEN"])
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "A-240242/legal-gqa-checkpoints" 
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"

In [4]:
def load_phase2_file(filename):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"Phase_2_LNER/Train/{filename}", repo_type="dataset", token=HF_TOKEN)
    if filename.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f.readlines() if line.strip()]
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 2 Data...")
# Load and merge text dictionaries
text_data = load_phase2_file("fold1-text.json")
text_data.update(load_phase2_file("fold2-text.json"))

# Load and merge gold annotation dictionaries
gold_data = load_phase2_file("fold1-gold.json")
gold_data.update(load_phase2_file("fold2-gold.json"))

entity_types = load_phase2_file("labels.txt")
print(f"Loaded {len(text_data)} total documents across both folds.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Loaded 70 total documents across both folds.


In [5]:
# 1. Build BIO Schema
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

# 2. Load Phase 1 Tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"

# 3. Manual Sliding Window Alignment
def align_spans_to_tokens(raw_text, gold_spans, tokenizer, max_length=1024, stride=256):
    # Tokenize the ENTIRE document without truncation
    tokenized_full = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
    full_ids = tokenized_full["input_ids"]
    full_offsets = tokenized_full["offset_mapping"]
    
    chunked_data = {"input_ids": [], "attention_mask": [], "labels": []}
    step = max_length - stride
    
    # Manually slide a window across the massive token arrays
    for i in range(0, len(full_ids), step):
        chunk_ids = full_ids[i : i + max_length]
        chunk_offs = full_offsets[i : i + max_length]
        
        pad_len = max_length - len(chunk_ids)
        chunk_ids = chunk_ids + [tokenizer.pad_token_id] * pad_len
        chunk_offs = chunk_offs + [(0, 0)] * pad_len
        chunk_mask = [1] * (max_length - pad_len) + [0] * pad_len
        
        chunk_labels = []
        current_entity_id = None
        
        for tok_start, tok_end in chunk_offs:
            if tok_start == tok_end:
                chunk_labels.append(-100)
                continue

            sub_text = raw_text[tok_start:tok_end]
            clean_tok_start = tok_start + (len(sub_text) - len(sub_text.lstrip()))
            clean_tok_end = tok_end - (len(sub_text) - len(sub_text.rstrip()))

            matched_label, matched_span_idx = "O", None
            
            for s_idx, span in enumerate(gold_spans):
                g_start, g_end, g_label = span[:3] 
                if not (clean_tok_end <= g_start or clean_tok_start >= g_end):
                    matched_label = g_label
                    matched_span_idx = s_idx
                    break

            if matched_label == "O":
                chunk_labels.append(LABEL2ID["O"])
                current_entity_id = None
            else:
                if current_entity_id == matched_span_idx:
                    chunk_labels.append(LABEL2ID[f"I-{matched_label}"])
                else:
                    chunk_labels.append(LABEL2ID[f"B-{matched_label}"])
                    current_entity_id = matched_span_idx
                    
        chunked_data["input_ids"].append(chunk_ids)
        chunked_data["attention_mask"].append(chunk_mask)
        chunked_data["labels"].append(chunk_labels)

    return chunked_data

tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

In [6]:
print("Processing and Aligning Training Spans...")
all_features = {"input_ids": [], "attention_mask": [], "labels": []}

for doc_id, raw_text in text_data.items():
    gold_spans = gold_data.get(doc_id, [])
    aligned = align_spans_to_tokens(raw_text, gold_spans, tokenizer)
    
    all_features["input_ids"].extend(aligned["input_ids"])
    all_features["attention_mask"].extend(aligned["attention_mask"])
    all_features["labels"].extend(aligned["labels"])

train_dataset = Dataset.from_dict(all_features)
print(f"Successfully chunked into {len(train_dataset)} training sequences.")

Processing and Aligning Training Spans...
Successfully chunked into 636 training sequences.


In [23]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

class GQAConfig(PretrainedConfig):
    model_type = "gqa_causal_lm"
    def __init__(self, vocab_size=32000, d_model=768, num_layers=8, num_heads=12, num_kv_heads=4, d_ff=2048, pad_token_id=0, eos_token_id=0, **kwargs):
        super().__init__(pad_token_id=pad_token_id, eos_token_id=eos_token_id, **kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_ff = d_ff

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(variance + self.eps))

class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.num_kv_heads = config.num_kv_heads
        self.head_dim = config.d_model // config.num_heads
        self.num_groups = self.num_heads // self.num_kv_heads
        
        self.q_proj = nn.Linear(config.d_model, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.d_model, bias=False)
        
    def forward(self, x):
        B, S, _ = x.size()
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        k = k.repeat_interleave(self.num_groups, dim=1)
        v = v.repeat_interleave(self.num_groups, dim=1)
        
        # Bidirectional Context for Token Classification
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.o_proj(attn.transpose(1, 2).contiguous().view(B, S, -1))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.fc2 = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.act = nn.GELU()
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

# --- HOULSBY ADAPTER IMPLEMENTATION ---
class HoulsbyAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_size=64):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck_size)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck_size, d_model)
        
        # Initialize to near-identity to prevent early training collapse
        nn.init.normal_(self.down.weight, std=1e-3)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.act(self.down(x)))

class GQABlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = GroupedQueryAttention(config)
        self.adapter1 = HoulsbyAdapter(config.d_model, bottleneck_size=64)
        self.mlp = MLP(config)
        self.adapter2 = HoulsbyAdapter(config.d_model, bottleneck_size=64)
        self.norm1 = RMSNorm(config.d_model)
        self.norm2 = RMSNorm(config.d_model)
        
    def forward(self, x):
        # Sequential Injection 1: After Attention
        attn_out = self.attn(self.norm1(x))
        x = x + attn_out + self.adapter1(attn_out)
        
        # Sequential Injection 2: After MLP
        mlp_out = self.mlp(self.norm2(x))
        x = x + mlp_out + self.adapter2(mlp_out)
        return x

class GQALightweightLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.pos_embedding = nn.Embedding(2048, config.d_model)
        self.layers = nn.ModuleList([GQABlock(config) for _ in range(config.num_layers)])
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
    def forward(self, input_ids, labels=None, **kwargs):
        B, S = input_ids.size()
        positions = torch.arange(S, device=input_ids.device).unsqueeze(0).expand(B, -1)
        x = self.embedding(input_ids) * math.sqrt(self.config.d_model) + self.pos_embedding(positions)
        for layer in self.layers: x = layer(x)
        logits = self.lm_head(self.norm(x))
        
        loss = None
        if labels is not None:
            class_weights = torch.ones(logits.size(-1), device=logits.device) * 3.0
            class_weights[0] = 1.0  
            loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        from transformers.modeling_outputs import TokenClassifierOutput
        return TokenClassifierOutput(loss=loss, logits=logits)

print("Initializing GQA Architecture for Houlsby Tuning...")
config = GQAConfig(vocab_size=len(tokenizer), pad_token_id=tokenizer.pad_token_id, num_layers=8, num_kv_heads=4)
model = GQALightweightLM(config)

print("Downloading and Loading Phase 1 Weights...")
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="model.safetensors", token=HF_TOKEN)
model.load_state_dict(load_file(model_weight_path), strict=False)

# Reinitialize head for 25 BIO classes
model.lm_head = nn.Linear(config.d_model, NUM_LABELS, bias=True)

# 1. Freeze EVERYTHING first
for param in model.parameters(): param.requires_grad = False

# 2. Unfreeze ONLY the sequential Houlsby Adapters
for layer in model.layers:
    for param in layer.adapter1.parameters(): param.requires_grad = True
    for param in layer.adapter2.parameters(): param.requires_grad = True

# 3. Keep Classification Head unfrozen
for param in model.lm_head.parameters(): param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready. Houlsby parameters: {trainable_params:,}")

Initializing GQA Architecture for Houlsby Tuning...
Model ready. Houlsby parameters: 1,605,401


In [27]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./phase2_gqa_lorma",
    report_to="none",
    num_train_epochs=30,          # Give it time to adapt to bidirectional context
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-3,          
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Commencing LORMA adaptation for L-NER...")
trainer.train()

Commencing LORMA adaptation for L-NER...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,4.286838
100,3.356904
150,3.183878
200,2.952564
250,2.841591
300,2.724434
350,2.624603
400,2.554613
450,2.469513
500,2.408983


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=600, training_loss=2.838025665283203, metrics={'train_runtime': 1114.0492, 'train_samples_per_second': 17.127, 'train_steps_per_second': 0.539, 'total_flos': 0.0, 'train_loss': 2.838025665283203, 'epoch': 30.0})

In [9]:
# 4. Inference Decoding Function with Relaxed Span Merger
def decode_token_predictions_to_spans(raw_text, chunks_preds, chunks_offsets):
    extracted_spans = []

    for preds, offsets in zip(chunks_preds, chunks_offsets):
        current_entity = None 

        for pred_id, (tok_start, tok_end) in zip(preds, offsets):
            if tok_start == tok_end:
                continue

            tag = ID2LABEL.get(pred_id, "O")

            if tag.startswith("B-"):
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                current_entity = {"label": tag[2:], "start": tok_start, "end": tok_end}

            elif tag.startswith("I-"):
                label_type = tag[2:]
                if current_entity is not None and current_entity["label"] == label_type:
                    current_entity["end"] = tok_end
                else:
                    if current_entity is not None:
                        extracted_spans.append(current_entity)
                    current_entity = {"label": label_type, "start": tok_start, "end": tok_end}
            else: 
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                    current_entity = None

        if current_entity is not None:
            extracted_spans.append(current_entity)

   # --- THE FIX: Smart Gap Merger ---
    merged_spans = []
    for span in extracted_spans:
        if not merged_spans:
            merged_spans.append(span)
            continue
        
        prev = merged_spans[-1]
        gap = span["start"] - prev["end"]
        
        if span["label"] == prev["label"] and 0 <= gap <= 12:
            # Extract the exact text between the two fragments
            gap_text = raw_text[prev["end"]:span["start"]].lower().strip()
            
            # 1. Is the gap purely whitespace, punctuation, OR numbers (to catch missed months like ".10.")?
            is_safe_gap = all(c in ".,/- &0123456789" for c in gap_text)
            
            # 2. Is the gap a common legal connector?
            is_connector = gap_text in ["", "of", "v", "v.", "vs", "vs.", "and", "&"]
            
            if is_safe_gap or is_connector:
                prev["end"] = span["end"] 
                continue # Safely stitched!
            
        merged_spans.append(span)

    # Clean, Snap Boundaries, and Deduplicate
    final_spans = []
    seen = set()

    for span in merged_spans:
        s, e, label = span["start"], span["end"], span["label"]
        
        clean_start = s
        clean_end = e
        
        while clean_start < clean_end and (raw_text[clean_start].isspace() or raw_text[clean_start] in ".,;:-"):
            clean_start += 1
        while clean_end > clean_start and (raw_text[clean_end-1].isspace() or raw_text[clean_end-1] in ".,;:-"):
            clean_end -= 1

        if clean_start >= clean_end:
            continue

        span_tuple = (clean_start, clean_end, label)
        if span_tuple not in seen:
            seen.add(span_tuple)
            final_spans.append([clean_start, clean_end, label])

    final_spans.sort(key=lambda x: (x[0], x[1]))
    return final_spans

In [10]:
# # --- Verification Test ---
# test_doc = "IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 1128 OF 2026 RAMESH ... APPELLANT"
# test_gold = [
#     [7, 29, "COURT"],      # "SUPREME COURT OF INDIA"
#     [62, 98, "CASENO"],     # "CRIMINAL APPEAL NO. 1128 OF 2026"
#     [99, 105, "APP"]        # "RAMESH"
# ]

# # 1. Encode
# features = align_spans_to_tokens(test_doc, test_gold, tokenizer, max_length=128, stride=32)

# # 2. Simulate perfect model predictions by re-using gold label targets
# simulated_preds = [
#     [lbl if lbl != -100 else 0 for lbl in chunk_lbls] 
#     for chunk_lbls in features["labels"]
# ]

# # 3. Decode
# recovered_spans = decode_token_predictions_to_spans(
#     test_doc, 
#     features["input_ids"], 
#     features["offset_mapping"], 
#     simulated_preds
# )

# # 4. Assert Exact Character-Level Match
# print("Original Gold Spans: ", test_gold)
# print("Recovered Spans:     ", recovered_spans)

# assert recovered_spans == test_gold, "Alignment failed! Offsets do not match exactly."
# print("\nAlignment pipeline verified: 100% character-offset correctness achieved.")

In [28]:
# --- Cell 6: Bulletproof Inference & CodaBench Submission ---
import zipfile
import json
import torch
import gc
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
import os

print("Loading Test Data...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_2_LNER/Test/fold3-text.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Successfully loaded {len(test_data)} documents from the test fold.")

# --- THE FIX: Rebuild ID2LABEL locally in memory ---
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

model.eval()
model.to("cuda")
predictions = {}

for doc_id, raw_text in tqdm(test_data.items(), desc="Generating L-NER Spans"):
    torch.cuda.empty_cache()
    gc.collect()
    
    try:
        # 1. Tokenize document without truncation
        tokenized = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
        full_ids = tokenized["input_ids"]
        full_offsets = tokenized["offset_mapping"]
        
        chunk_size = 1024
        stride = 256
        step = chunk_size - stride
        
        doc_preds = []
        doc_offsets = []
        
        # 2. Slide window across document
        for i in range(0, len(full_ids), step):
            chunk_ids = full_ids[i : i + chunk_size]
            chunk_offs = full_offsets[i : i + chunk_size]
            
            input_tensor = torch.tensor([chunk_ids]).to("cuda")
            
            # 3. Model Prediction with Class Penalty
            with torch.inference_mode():
                outputs = model(input_ids=input_tensor)
                logits = outputs.logits
                #logits[:, :, 0] -= 3.0  # Penalize 'O' class
                preds = torch.argmax(logits, dim=-1).cpu().numpy()[0]
                
            doc_preds.append(preds)
            doc_offsets.append(chunk_offs)
            
        # 4. Decode tokens back to exact character spans
        extracted_spans = decode_token_predictions_to_spans(raw_text, doc_preds, doc_offsets)
        predictions[doc_id] = extracted_spans

    except Exception as e:
        print(f"\nError processing document {doc_id}: {str(e)}")
        predictions[doc_id] = []

# 5. Save to Submission Format
os.makedirs("./phase2_gqa_lorma", exist_ok=True)
PREDICTION_PATH = "./phase2_gqa_lorma/prediction.json"
SUBMISSION_ZIP = "./phase2_gqa_lorma/submission.zip"

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nFinal sanity check: Saved {len(predictions.keys())} document IDs to prediction.json")
print(f"L-NER Predictions complete! Submission saved to: {SUBMISSION_ZIP}")

Loading Test Data...
Successfully loaded 35 documents from the test fold.


Generating L-NER Spans:   0%|          | 0/35 [00:00<?, ?it/s]


Final sanity check: Saved 35 document IDs to prediction.json
L-NER Predictions complete! Submission saved to: ./phase2_gqa_lorma/submission.zip


In [29]:
# Run this in a new cell to diagnose boundary artifacts
first_doc_id = list(predictions.keys())[0]
raw_doc_text = test_data[first_doc_id]

print(f"--- Diagnostic: Top 20 Predictions for Doc {first_doc_id} ---")
for span in predictions[first_doc_id][:20]:
    start, end, label = span
    extracted_text = raw_doc_text[start:end]
    # Using repr() to expose hidden trailing spaces or newlines (e.g. 'Court \n')
    print(f"[{label: <10}] : {repr(extracted_text)}")

--- Diagnostic: Top 20 Predictions for Doc 1128835 ---
[CASENO    ] : 'APPEAL'
[DATE      ] : '6'
[DATE      ] : '2008'
[APP       ] : 'SING'
[APP       ] : 'SPECIAL'
[CASENO    ] : ')'
[CASENO    ] : '154'
[CASENO    ] : '2004'
[APP       ] : 'AB'
[CASENO    ] : 'S'
[CASENO    ] : 'LTD'
[CASENO    ] : 'ANR'
[CASENO    ] : 'S'
[CASENO    ] : 'ANR'
[RESP      ] : 'WITH'
[CASENO    ] : 'APPEAL'
[DATE      ] : '6'
[CASENO    ] : '58'
[DATE      ] : '2008'
[APP       ] : 'SING'
